In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

model_df = pd.read_csv('f1_2023_lap_model_data.csv')

C:\Users\bhavi\AppData\Local\Temp\ipykernel_34388\3876853436.py:5: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  model_df = pd.read_csv('f1_2023_lap_model_data.csv')


In [3]:
track_df = pd.read_csv('track_char.csv')
track_df['EventName'] = track_df['EventName'].str.strip()
track_df['TrackDirection'] = track_df['TrackDirection'].str.strip().str.capitalize()

model_df = model_df.merge(track_df, on='EventName', how='left')

print(model_df[track_df.columns].isna().sum())

EventName            0
CircuitLength_km     0
NumCorners           0
NumDRSZones          0
TrackDirection       0
AvgSpeed_kmh         0
ElevationChange_m    0
DownforceLevel       0
dtype: int64


In [4]:
stint_median = model_df.groupby(['EventName', 'Driver', 'Stint'])['LapTime_Seconds'].transform('median')

model_df['is_anomalous_lap'] = model_df['LapTime_Seconds'] > (1.07 * stint_median)

# 1. Update clean-lap filter
model_df['is_clean_lap'] = (model_df['is_green_flag'] & 
                              ~model_df['is_pit_lap'] & 
                              (model_df['IsAccurate'] == True) &
                              ~model_df['is_anomalous_lap'])

# 2. Re-filter
model_df = model_df[model_df['is_clean_lap'] == True].copy()

# 3. Re-sort and re-derive the degradation target on the now-cleaned data
model_df = model_df.sort_values(['EventName', 'Driver', 'Stint', 'LapNumber'])
stint_baseline = model_df.groupby(['EventName', 'Driver', 'Stint'])['LapTime_Seconds'].transform('first')
model_df['DegradationDelta_Secs'] = model_df['LapTime_Seconds'] - stint_baseline

# 4. Re-check the distribution
print(model_df['DegradationDelta_Secs'].describe())
model_df = model_df[model_df['is_clean_lap'] == True].copy()
model_df = model_df.dropna(subset=['LapTime_Seconds', 'TyreLife', 'TrackTemp', 'AirTemp'])

count    20357.000000
mean        -0.687166
std          1.719869
min        -16.566000
25%         -1.387000
50%         -0.309000
75%          0.264000
max          7.896000
Name: DegradationDelta_Secs, dtype: float64


In [5]:
print(model_df[(model_df['EventName']=='Mexico City Grand Prix') & 
               (model_df['Driver']=='PIA') & 
               (model_df['LapNumber']==36)][['TrackStatus', 'IsAccurate', 'Deleted', 'DeletedReason', 'PitInTime', 'PitOutTime']])

Empty DataFrame
Columns: [TrackStatus, IsAccurate, Deleted, DeletedReason, PitInTime, PitOutTime]
Index: []


In [6]:
model_df = model_df.sort_values(['EventName', 'Driver', 'Stint', 'LapNumber'])

stint_baseline = model_df.groupby(['EventName', 'Driver', 'Stint'])['LapTime_Seconds'].transform('first')
model_df['DegradationDelta_Secs'] = model_df['LapTime_Seconds'] - stint_baseline

In [7]:
sample = model_df[(model_df['EventName'] == 'Bahrain Grand Prix') & (model_df['Driver'] == 'VER')]
print(sample[['LapNumber', 'Stint', 'TyreLife', 'LapTime_Seconds', 'DegradationDelta_Secs']].head(20))

      LapNumber  Stint  TyreLife  LapTime_Seconds  DegradationDelta_Secs
841         3.0    1.0       6.0           98.006                  0.000
1149        4.0    1.0       7.0           97.976                 -0.030
1523        5.0    1.0       8.0           98.035                  0.029
1865        6.0    1.0       9.0           97.986                 -0.020
2198        7.0    1.0      10.0           98.021                  0.015
2588        8.0    1.0      11.0           98.154                  0.148
2933        9.0    1.0      12.0           98.278                  0.272
3235       10.0    1.0      13.0           98.369                  0.363
3556       11.0    1.0      14.0           98.483                  0.477
3842       12.0    1.0      15.0           98.591                  0.585
4153       13.0    1.0      16.0           98.482                  0.476
5159       16.0    2.0       2.0           97.801                  0.000
5469       17.0    2.0       3.0           97.648  

In [8]:
features = ['TyreLife', 'Compound', 'FreshTyre', 'TrackTemp', 'AirTemp', 'Driver',
            'CircuitLength_km', 'NumCorners', 'NumDRSZones', 'TrackDirection',
            'AvgSpeed_kmh', 'ElevationChange_m', 'DownforceLevel']

X = model_df[features].copy()
y = model_df['DegradationDelta_Secs']

X = pd.get_dummies(X, columns=['Compound', 'Driver', 'TrackDirection', 'DownforceLevel'])

In [9]:

sorted_events = model_df[['RoundNumber', 'EventName']].drop_duplicates().sort_values('RoundNumber')
event_order = sorted_events['EventName'].tolist()

n_test_races = 5
train_events = event_order[:-n_test_races]
test_events = event_order[-n_test_races:]

train_mask = model_df['EventName'].isin(train_events)
test_mask = model_df['EventName'].isin(test_events)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [10]:

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print(f"MAE: {mae:.3f} seconds")

MAE: 1.238 seconds


In [11]:
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(15))

TyreLife                 0.203672
NumCorners               0.109125
TrackTemp                0.104672
Compound_INTERMEDIATE    0.101971
ElevationChange_m        0.090709
AirTemp                  0.073881
CircuitLength_km         0.035789
Driver_ALB               0.020897
Driver_PER               0.019800
Driver_ZHO               0.019086
Driver_HUL               0.017382
AvgSpeed_kmh             0.016429
Compound_HARD            0.016287
Driver_STR               0.015568
Driver_SAR               0.013939
dtype: float64


In [12]:
import joblib
joblib.dump(model, 'tire_deg.joblib')

['tire_deg.joblib']